# Датасет BDD100K: сборка и обзор

Этот ноутбук **не содержит логики**. Вся работа делается функциями из `src/`,
здесь только вызовы и картинки.

Так и должно быть в проекте: код живёт в `.py`-модулях, которые можно
импортировать, тестировать и запускать из `train.py`, а ноутбук — это витрина
поверх них. Если завтра поменяется формат разметки, правка будет в одном месте
(`src/datasets/bdd100k.py`), а не в трёх ноутбуках, которые давно разошлись
между собой.

Что здесь происходит:

1. установка окружения из `requirements.txt`;
2. сборка датасета в формате YOLO (`bdd100k.build_yolo_dataset`);
3. информация о датасете — распределение классов и баланс день/ночь;
4. отрисовка пары сцен с разметкой.

Шаг 4 — не украшение. Ошибка в конвертации боксов не вызывает исключений:
обучение пойдёт, лосс будет падать, метрика останется около нуля. Глазами это
видно за секунду, по логам — нет.

## 1. Окружение

`torch` намеренно не закреплён по точной версии: на Kaggle и в облачных
образах уже стоит сборка, слинкованная с CUDA, и жёсткий пин заставил бы pip
переустановить её поверх — часто на CPU-версию.

In [ ]:
import subprocess
import sys
from pathlib import Path


def find_repo_root(start=None):
    """Ищет корень репозитория по наличию src/datasets/bdd100k.py."""
    here = Path(start or Path.cwd()).resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "src" / "datasets" / "bdd100k.py").exists():
            return candidate
    raise FileNotFoundError(
        "Не нашёл корень проекта. Запускайте ноутбук из репозитория, "
        "а на Kaggle сначала склонируйте его:\n"
        "  !git clone https://github.com/Antonoof/Yandex-night-vision-Detection.git"
    )


ROOT = find_repo_root()
sys.path.insert(0, str(ROOT))  # чтобы заработал `import src...`
print("корень проекта:", ROOT)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")],
    check=True,
)
print("зависимости установлены")

In [ ]:
import logging

import matplotlib.pyplot as plt
import numpy as np

from src.datasets import bdd100k
from src.utils.visualize import draw_ground_truth

# describe_split_balance пишет таблицу через logging, а не print —
# так один и тот же код одинаково работает и в ноутбуке, и в train.py
logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)

# --- что смотрим ---
INPUT_DIR = Path("/kaggle/input") if Path("/kaggle/input").exists() else ROOT / "data"
WORK_DIR = ROOT / "data" / "bdd_yolo"   # куда собрать YOLO-датасет
MAX_TRAIN_IMAGES = None                 # None = весь train; число = быстрая проба
SEED = 42

## 2. Читаем разметку

`find_dataset_root` ищет папку, где рядом лежат `images/` и `val.json`, —
путь монтирования на Kaggle нестабилен, поэтому его не хардкодят.

`load_records` превращает `<split>.json` в единый формат: на каждый кадр —
имя, путь, время суток и список боксов `(класс, cx, cy, w, h)` в нормированных
координатах. Заодно он сверяет реальный размер первого кадра с константой
`IMG_W × IMG_H = 1280 × 720`: если датасет окажется другого разрешения,
денормализация боксов молча испортит все метрики, поэтому лучше упасть сразу.

`test` намеренно не читаем — он остаётся нетронутым до конца проекта.

In [ ]:
data_root = bdd100k.find_dataset_root(INPUT_DIR)
print("датасет:", data_root)

train_records = bdd100k.load_records(data_root, "train")
val_records = bdd100k.load_records(data_root, "val")

stats = bdd100k.describe_split_balance(train_records, val_records)

## 3. Распределение классов

Две картины, которые определяют, как читать любые будущие метрики.

In [ ]:
# Цвета: слоты 1 и 2 категориальной палитры, в фиксированном порядке.
NIGHT, DAY = "#2a78d6", "#eb6834"
INK, INK_MUTED, SURFACE = "#0b0b0b", "#52514e", "#fcfcfb"

names = list(stats["per_class"])
night = np.array([stats["per_class"][n]["night"] for n in names])
day = np.array([stats["per_class"][n]["day"] for n in names])
share = np.array([stats["per_class"][n]["night_share"] for n in names])
y = np.arange(len(names))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.6), facecolor=SURFACE)

# --- слева: сколько боксов каждого класса (log: разброс в тысячи раз) ---
ax1.barh(y - 0.21, night, height=0.38, color=NIGHT, label="ночь")
ax1.barh(y + 0.21, day, height=0.38, color=DAY, label="день")
ax1.set_xscale("log")
ax1.set_xlabel("боксов, шт (логарифмическая шкала)", color=INK_MUTED, fontsize=9)
ax1.set_title("Сколько объектов каждого класса", color=INK, fontsize=11, loc="left")
ax1.set_ylim(len(names) - 0.4, -1.4)
ax1.legend(frameon=False, fontsize=9, loc="lower right")

# --- справа: какая доля класса снята ночью ---
ax2.barh(y, share * 100, height=0.62, color=NIGHT)
overall = night.sum() / (night.sum() + day.sum()) * 100
ax2.axvline(overall, color=INK_MUTED, linestyle="--", linewidth=1, zorder=1)
# подпись линии — над верхним столбцом, иначе она садится на подписи оси
ax2.text(overall + 0.6, -0.9, f"в среднем {overall:.0f}%",
         color=INK_MUTED, fontsize=8, va="bottom", ha="left")
for yi, s in zip(y, share):
    # отступ с запасом, чтобы число не садилось на пунктир
    # подложка под цвет фона: подпись может лечь поверх пунктира
    ax2.text(s * 100 + 1.2, yi, f"{s:.1%}", va="center", ha="left",
             color=INK_MUTED, fontsize=8, zorder=3,
             bbox=dict(facecolor=SURFACE, edgecolor="none", pad=1.0))
ax2.set_xlim(0, max(share) * 100 * 1.32)
ax2.set_ylim(len(names) - 0.4, -1.4)   # место сверху под подпись линии
ax2.set_xlabel("доля ночных боксов, %", color=INK_MUTED, fontsize=9)
ax2.set_title("Насколько класс «ночной»", color=INK, fontsize=11, loc="left")

for ax in (ax1, ax2):
    ax.set_facecolor(SURFACE)
    ax.set_yticks(y, names, fontsize=9, color=INK)
    ax.tick_params(colors=INK_MUTED, labelsize=8)
    ax.grid(axis="x", color="#e6e5e1", linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right", "left"):
        ax.spines[side].set_visible(False)
    ax.spines["bottom"].set_color("#d6d5d0")

plt.tight_layout()
plt.show()

### Что из этого следует

**Дисбаланс классов огромный, и шкала слева логарифмическая не случайно** —
между `car` и `train` разница в тысячи раз. Практическое следствие: mAP
усредняется **по классам, а не по объектам**, поэтому класс с сотней боксов
весит в метрике столько же, сколько класс с полумиллионом. Один невыученный
редкий класс отнимает у общей метрики целую восьмую часть.

**Ночь недопредставлена** — примерно каждый шестой бокс. Это одна из причин,
по которой детекторы хуже работают ночью: дело не только в темноте, но и в
том, что ночных примеров модель видела мало.

**Правая панель показывает, что «ночной» класс — понятие относительное.**
У `traffic light` доля ночи заметно выше среднего, у `person` — заметно ниже.
То есть ночью в кадре больше светофоров и меньше пешеходов, и выучиваются эти
классы по-разному.

## 4. Собираем датасет в формате YOLO

`ultralytics` не умеет читать нашу разметку напрямую. Ему нужно дерево, где
рядом с `images/` лежит `labels/`, и каждому `123.jpg` соответствует
`123.txt` с одной строкой на объект: `<класс> <cx> <cy> <ширина> <высота>`.

Кадры без объектов тоже попадают в датасет — с пустым `.txt`. Такие «фоновые»
картинки учат модель не выдумывать объекты на пустом месте.

Картинки не копируются, а линкуются символическими ссылками: исходная папка
доступна только на чтение, а копия десятков тысяч файлов — это лишние
гигабайты. Если файловая система симлинки не поддерживает, код молча
переключается на копирование.

In [ ]:
train_used = bdd100k.subsample(train_records, MAX_TRAIN_IMAGES, SEED)

# Сплиты независимы по построению, но проверка стоит миллисекунды,
# а незамеченная утечка обесценила бы все метрики проекта.
assert not ({r["name"] for r in train_used} & {r["name"] for r in val_records}), \
    "один и тот же кадр есть и в train, и в val!"

data_yaml = bdd100k.build_yolo_dataset(
    {"train": train_used, "val": val_records}, WORK_DIR
)

for split in ("train", "val"):
    n_img = len(list((WORK_DIR / "images" / split).glob("*.jpg")))
    n_lbl = len(list((WORK_DIR / "labels" / split).glob("*.txt")))
    print(f"{split:5s}: {n_img} картинок, {n_lbl} файлов разметки "
          f"{'OK' if n_img == n_lbl else 'РАСХОЖДЕНИЕ!'}")

print("\ndata.yaml:\n" + data_yaml.read_text())

## 5. Отрисовка сцен

Проверяем разметку глазами: если рамки лежат на объектах — конвертация
`cx, cy, w, h` → пиксельные координаты верна.

Берём одну ночную сцену и одну дневную, причём **фиксированные**
(первые по имени, а не случайные): так картинку можно сравнивать между
запусками, а не смотреть каждый раз на новые кадры.

In [ ]:
night_val = sorted((r for r in val_records if r["timeofday"] == "night"),
                   key=lambda r: r["name"])
day_val = sorted((r for r in val_records if r["timeofday"] == "daytime"),
                 key=lambda r: r["name"])
print(f"в val: {len(night_val)} ночных кадров, {len(day_val)} дневных")

draw_ground_truth([night_val[0], day_val[0]])
plt.show()

## Дальше

Датасет собран и проверен, `data.yaml` готов. Обучение запускается **не
отсюда**, а из командной строки, конфигом:

```bash
python3 train.py                              # полный baseline
python3 train.py trainer.epochs=1 datasets.max_train_images=200   # быстрая проба
```

Что логируется в Comet и по каким правилам называются запуски — в
[docs/EXPERIMENTS.md](../docs/EXPERIMENTS.md).